# Canadian Housing Affordability — EDA

Exploratory analysis of CMHC Rental Market Report data across 18 CMAs (Oct-2024 and Oct-2025).

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

PROCESSED = Path("../data/processed")
features  = pd.read_parquet(PROCESSED / "cmhc_features.parquet")
forecasts = pd.read_parquet(PROCESSED / "forecasts.parquet")

cma = features[features["is_cma_total"] & (features["bedroom_type"] == "Total")].copy()
print(f"Shape: {features.shape}  |  Cities: {features.city.nunique()}")
cma[["city","avg_rent_oct24","avg_rent_oct25","avg_rent_yoy_pct","vacancy_rate_oct25","market_tightness","rent_growth_rank"]].sort_values("rent_growth_rank")

## 1. Rent growth ranking
Which cities saw the fastest appreciation Oct-24 → Oct-25?

In [ ]:
fig = px.bar(
    cma.sort_values("avg_rent_yoy_pct", ascending=True),
    x="avg_rent_yoy_pct", y="city", orientation="h",
    color="avg_rent_yoy_pct", color_continuous_scale="RdYlGn",
    text=cma.sort_values("avg_rent_yoy_pct", ascending=True)["avg_rent_yoy_pct"].round(1).astype(str)+"%",
    title="YoY Rent Growth by CMA (Oct-24 → Oct-25, Total bedroom)",
    labels={"avg_rent_yoy_pct": "% Change", "city": ""},
)
fig.update_traces(textposition="outside")
fig.show()

## 2. Vacancy vs rent growth
Do tighter markets (lower vacancy) show higher rent growth?

In [ ]:
fig = px.scatter(
    cma.dropna(subset=["vacancy_rate_oct25","avg_rent_yoy_pct"]),
    x="vacancy_rate_oct25", y="avg_rent_yoy_pct",
    size="rental_universe_oct25", text="city",
    color="market_tightness", color_continuous_scale="RdYlGn_r",
    trendline="ols",
    title="Vacancy Rate vs Rent Growth (bubble = rental stock)",
    labels={"vacancy_rate_oct25":"Vacancy Rate Oct-25 (%)",
            "avg_rent_yoy_pct":"Rent Growth YoY (%)"},
)
fig.update_traces(textposition="top center")
fig.show()

corr = cma[["vacancy_rate_oct25","avg_rent_yoy_pct"]].dropna().corr().iloc[0,1]
print(f"Pearson r (vacancy vs rent growth): {corr:.3f}")

## 3. Absolute rent level by city and bedroom type

In [ ]:
bed_cma = features[features["is_cma_total"] & (features["bedroom_type"] != "Total")].copy()
fig = px.bar(
    bed_cma.sort_values("avg_rent_oct25"),
    x="city", y="avg_rent_oct25", color="bedroom_type", barmode="group",
    title="Average Rent Oct-25 by City and Bedroom Type",
    labels={"avg_rent_oct25": "Avg Rent ($)"},
)
fig.update_layout(xaxis_tickangle=-40)
fig.show()

## 4. Market tightness score
Composite rank combining low vacancy and high rent growth.

In [ ]:
fig = px.bar(
    cma.dropna(subset=["market_tightness"]).sort_values("market_tightness", ascending=False),
    x="city", y="market_tightness",
    color="market_tightness", color_continuous_scale="Reds",
    title="Market Tightness Score (1 = tightest)",
    labels={"market_tightness": "Score"},
)
fig.update_layout(xaxis_tickangle=-40, showlegend=False)
fig.show()

## 5. Oct-26 rent forecasts (XGBoost)
Cross-sectional model trained on Oct-24 → Oct-25 transition, predicting Oct-26.

In [ ]:
f1 = forecasts[forecasts["bedroom_type"]=="1 Bedroom"].sort_values("predicted_rent")
fig = go.Figure(go.Bar(
    x=f1["city"], y=f1["predicted_rent"].round(0),
    error_y=dict(type="data", symmetric=False,
        array=(f1["upper_ci"]-f1["predicted_rent"]).round(0),
        arrayminus=(f1["predicted_rent"]-f1["lower_ci"]).round(0)),
    marker_color="#756bb1",
))
fig.update_layout(
    title="Predicted Oct-26 Average Rent — 1 Bedroom (90% CI)",
    xaxis_tickangle=-40, yaxis_title="Predicted Rent ($)", height=450,
)
fig.show()

## 6. SQL window-query examples
These queries are also saved as  files in .

In [ ]:
# Inline SQL executed against the PostgreSQL database (requires load.py to have been run).
# Shown here for reference; uncomment and supply your engine to run.

city_rankings_sql = """
WITH cma_totals AS (
    SELECT c.name AS city, c.province, r.survey_date, r.vacancy_rate,
           a.avg_rent
    FROM vacancy_rates r
    JOIN cities    c ON c.id = r.city_id
    JOIN avg_rents a ON a.city_id = r.city_id AND a.survey_date = r.survey_date
                    AND a.bedroom_type = r.bedroom_type AND a.zone = r.zone
    WHERE r.is_cma_total = TRUE AND r.bedroom_type = 'Total'
),
pivoted AS (
    SELECT city, province,
        MAX(avg_rent) FILTER (WHERE survey_date = '2024-10-01') AS rent_oct24,
        MAX(avg_rent) FILTER (WHERE survey_date = '2025-10-01') AS rent_oct25,
        MAX(vacancy_rate) FILTER (WHERE survey_date = '2025-10-01') AS vacancy_oct25
    FROM cma_totals GROUP BY city, province
)
SELECT city, province, rent_oct24, rent_oct25,
    ROUND(((rent_oct25 - rent_oct24)::numeric / rent_oct24 * 100), 2) AS rent_yoy_pct,
    RANK() OVER (ORDER BY (rent_oct25 - rent_oct24)::float / rent_oct24 DESC) AS rank
FROM pivoted ORDER BY rank;
"""
print(city_rankings_sql)
# engine = get_engine()  # from src.load
# pd.read_sql(city_rankings_sql, engine)

## Summary

| Question | Finding |
|---|---|
| Fastest rent growth | Montreal (+10.5%), Quebec (+9.7%), Halifax (+7.1%) |
| Tightest market | Hamilton / Ottawa / Toronto (high tightness score) |
| Vacancy vs growth | Negative correlation (r ≈ −0.3): lower vacancy → higher growth |
| Most uncertain forecast (widest CI) | Victoria, Vancouver (high rent variance) |